In [29]:
import os

intermediates = [
    "district_table_2026-09-01.csv",
    "district_validation_2026-09-01.csv",
    "state_analysis_2026-09-01.csv",
    "localities_2026-09-01.csv",
    "death_notes_2026-09-01.csv",
    "cross_validation_2026-09-01.csv",
    "locality_count_validation_2026-09-01.csv",
]

for f in intermediates:
    # Use ".." to step back to project root from the notebooks directory
    path = os.path.join("..", "data", "processed", f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {f}")
    else:
        print(f"File not found: {f}")

File not found: district_table_2026-09-01.csv
File not found: district_validation_2026-09-01.csv
File not found: state_analysis_2026-09-01.csv
File not found: localities_2026-09-01.csv
File not found: death_notes_2026-09-01.csv
File not found: cross_validation_2026-09-01.csv
File not found: locality_count_validation_2026-09-01.csv


In [30]:
import os
import sqlite3

db_path = "data/idsp_kerala.db"

# Close the existing connection if it exists in memory
if "conn" in globals():
    try:
        conn.close()
    except NameError:
        pass

if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cur = conn.cursor()
print("Fresh database created")

Fresh database created


In [31]:
cur.executescript("""
CREATE TABLE IF NOT EXISTS reports (
    report_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_date TEXT NOT NULL UNIQUE,
    period_type TEXT NOT NULL,
    source_url TEXT,
    filename TEXT NOT NULL,
    file_hash TEXT,
    ingested_at TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS observations (
    obs_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id INTEGER NOT NULL,
    geography_level TEXT NOT NULL,
    district_code TEXT NOT NULL,
    district_name TEXT NOT NULL,
    disease TEXT NOT NULL,
    metric TEXT NOT NULL,
    subtype TEXT,
    value REAL NOT NULL,
    source_page INTEGER,
    FOREIGN KEY (report_id) REFERENCES reports(report_id)
);

CREATE TABLE IF NOT EXISTS locality_reports (
    loc_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id INTEGER NOT NULL,
    district_code TEXT NOT NULL,
    district_name TEXT NOT NULL,
    disease_raw TEXT,
    disease TEXT NOT NULL,
    district_reported_count REAL,
    locality_text TEXT NOT NULL,
    raw_text TEXT,
    source_page INTEGER,
    FOREIGN KEY (report_id) REFERENCES reports(report_id)
);

CREATE TABLE IF NOT EXISTS death_notes (
    death_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id INTEGER NOT NULL,
    district_code TEXT,
    district_name TEXT,
    disease_raw TEXT,
    disease TEXT,
    age INTEGER,
    sex TEXT,
    locality TEXT,
    death_date TEXT,
    parse_status TEXT NOT NULL,
    raw_text TEXT,
    source_page INTEGER,
    FOREIGN KEY (report_id) REFERENCES reports(report_id)
);

CREATE TABLE IF NOT EXISTS state_analysis (
    state_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id INTEGER NOT NULL,
    serial_number INTEGER,
    disease_raw TEXT,
    disease TEXT NOT NULL,
    subtype_raw TEXT,
    subtype TEXT,
    daily_suspected_cases REAL,
    daily_suspected_deaths REAL,
    daily_confirmed REAL,
    daily_deaths REAL,
    month_suspected_cases REAL,
    month_suspected_deaths REAL,
    month_confirmed REAL,
    month_deaths REAL,
    cumulative_suspected_cases REAL,
    cumulative_suspected_deaths REAL,
    cumulative_confirmed REAL,
    cumulative_deaths REAL,
    source_page INTEGER,
    FOREIGN KEY (report_id) REFERENCES reports(report_id)
);

CREATE TABLE IF NOT EXISTS diseases (
    canonical_name TEXT PRIMARY KEY,
    aliases TEXT
);
""")
conn.commit()
print("Tables created")

Tables created


In [32]:
import hashlib
import pandas as pd

pdf_path = r"C:\Users\vinee\rag_chatbot_kerala\data\raw\daily\IDSP-Daily-Report-01.09.2026.pdf"
with open(pdf_path, "rb") as f:
    file_hash = hashlib.sha256(f.read()).hexdigest()

cur.execute("""
    INSERT OR IGNORE INTO reports (report_date, period_type, filename, file_hash)
    VALUES (?, ?, ?, ?)
""", ("2026-09-01", "daily", "IDSP-Daily-Report-01.09.2026.pdf", file_hash))
conn.commit()

cur.execute("SELECT report_id FROM reports WHERE report_date = '2026-09-01'")
report_id = cur.fetchone()[0]

# --- Observations ---
obs = pd.read_csv(r"C:\Users\vinee\rag_chatbot_kerala\data\processed\trusted_district_observations_2026-09-01.csv")
for _, row in obs.iterrows():
    cur.execute("""
        INSERT INTO observations
        (report_id, geography_level, district_code, district_name,
         disease, metric, subtype, value, source_page)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (report_id, row["geography_level"], row["district_code"],
          row["district_name"], row["disease"], row["metric"],
          row.get("subtype"), row["value"], row["source_page"]))

# --- Localities ---
loc = pd.read_csv(r"C:\Users\vinee\rag_chatbot_kerala\data\processed\trusted_localities_2026-09-01.csv")
for _, row in loc.iterrows():
    cur.execute("""
        INSERT INTO locality_reports
        (report_id, district_code, district_name, disease_raw, disease,
         district_reported_count, locality_text, raw_text, source_page)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (report_id, row["district_code"], row["district_name"],
          row.get("disease_raw"), row["disease"],
          row.get("district_reported_count"),
          row["locality_text"], row.get("raw_text"), row["source_page"]))

# --- Death notes ---
deaths = pd.read_csv(r"C:\Users\vinee\rag_chatbot_kerala\data\processed\trusted_death_notes_2026-09-01.csv")
for _, row in deaths.iterrows():
    cur.execute("""
        INSERT INTO death_notes
        (report_id, district_code, district_name, disease_raw, disease,
         age, sex, locality, death_date, parse_status, raw_text, source_page)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (report_id, row.get("district_code"), row.get("district_name"),
          row.get("disease_raw"), row.get("disease"),
          row.get("age"), row.get("sex"), row.get("locality"),
          row.get("death_date"), row["parse_status"],
          row.get("raw_text"), row["source_page"]))

# --- State analysis ---
state = pd.read_csv(r"C:\Users\vinee\rag_chatbot_kerala\data\processed\trusted_state_analysis_2026-09-01.csv")
for _, row in state.iterrows():
    cur.execute("""
        INSERT INTO state_analysis
        (report_id, serial_number, disease_raw, disease, subtype_raw, subtype,
         daily_suspected_cases, daily_suspected_deaths, daily_confirmed, daily_deaths,
         month_suspected_cases, month_suspected_deaths, month_confirmed, month_deaths,
         cumulative_suspected_cases, cumulative_suspected_deaths,
         cumulative_confirmed, cumulative_deaths, source_page)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (report_id, row.get("serial_number"), row.get("disease_raw"),
          row["disease"], row.get("subtype_raw"), row.get("subtype"),
          row.get("daily_suspected_cases"), row.get("daily_suspected_deaths"),
          row.get("daily_confirmed"), row.get("daily_deaths"),
          row.get("month_suspected_cases"), row.get("month_suspected_deaths"),
          row.get("month_confirmed"), row.get("month_deaths"),
          row.get("cumulative_suspected_cases"), row.get("cumulative_suspected_deaths"),
          row.get("cumulative_confirmed"), row.get("cumulative_deaths"),
          row["source_page"]))

conn.commit()
print(f"Loaded {len(obs)} observations, {len(loc)} localities, {len(deaths)} death notes, {len(state)} state analysis rows")

Loaded 406 observations, 24 localities, 6 death notes, 26 state analysis rows


In [33]:
for table in ["reports", "observations", "locality_reports", "death_notes", "state_analysis"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]} rows")

reports: 1 rows
observations: 406 rows
locality_reports: 24 rows
death_notes: 6 rows
state_analysis: 26 rows
